
A) UNCTAD LSCI (country connectivity)

B) UNCTAD LSBCI (bilateral connectivity)

C) CEPII GeoDist (bilateral distance)

D)Bunker fuel price (macro year proxy)

USDA Open Ag Transport “Daily Bunker Fuel Prices” dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pycountry

import pandas as pd
import numpy as np
import pycountry
import re

transport_path = "/content/drive/MyDrive/is5126/US.TransportCosts_1107_20260206_072800.csv"
lsci_path      = "/content/drive/MyDrive/is5126/US.LSCI_20260314_120502.csv"
lsbci_path     = "/content/drive/MyDrive/is5126/US.LSBCI_20260314_120615.csv"
fuel_path      = "/content/drive/MyDrive/is5126/Daily_Bunker_Fuel_Prices_20260314.csv"
dist_path      = "/content/drive/MyDrive/is5126/dist_cepii.csv"

# ---------- ISO3 mapping (covers UN-style labels) - COUNTRY CODE MAPPING ----------
manual = {
  'Brunei Darussalam': 'BRN',
  'Cabo Verde': 'CPV',
  'China, Hong Kong SAR': 'HKG',
  'China, Macao SAR': 'MAC',
  'China, Taiwan Province of': 'TWN',
  'Congo, Dem. Rep. of the': 'COD',
  "Cote d'Ivoire": 'CIV',
  'Curacao': 'CUW',
  'Eswatini': 'SWZ',
  "Korea, Dem. People's Rep. of": 'PRK',
  "Lao People's Dem. Rep.": 'LAO',
  'Netherlands (Kingdom of the)': 'NLD',
  'Russian Federation': 'RUS',
  'Saint Barthelemy': 'BLM',
  'Saint Helena': 'SHN',
  'State of Palestine': 'PSE',
  'Switzerland, Liechtenstein': 'CHE',
  'Turkiye': 'TUR',
  'Venezuela (Bolivarian Rep. of)': 'VEN',
  "Dem. People's Rep. of Korea": "PRK",
  "Dem. Rep. of the Congo": "COD",
  "Netherlands Antilles": "ANT",
  "Reunion": "REU",
  "Serbia and Montenegro": "SCG",
  "Sudan (...2011)": "SDN",
  "United States Virgin Islands": "VIR",
  "Wallis and Futuna Islands": "WLF",
}

def to_iso3(label: str):
    if pd.isna(label): return None
    s = str(label).strip()
    if s in manual: return manual[s]
    # small normalizations
    repl = {
        "Bolivia (Plurinational State of)": "Bolivia",
        "Viet Nam": "Vietnam",
        "Côte d’Ivoire": "Cote d'Ivoire",
        "Côte d'Ivoire": "Cote d'Ivoire",
    }
    s2 = repl.get(s, s)
    try:
        return pycountry.countries.lookup(s2).alpha_3
    except:
        return None

# ---------- Load transport & melt ----------
t = pd.read_csv(transport_path)
year_cols = [c for c in t.columns if re.match(r"20\d{2}_Perunit", c)]
long = t.melt(
    id_vars=["Origin_Label","Destination_Label","Product_Code","Product_Label"],
    value_vars=year_cols,
    var_name="YearCol",
    value_name="Freight"
)
long["Year"] = long["YearCol"].str.extract(r"(20\d{2})").astype(int)
obs = long.dropna(subset=["Freight"]).copy()
obs["Freight"] = obs["Freight"].astype(float)

obs["iso_o"] = obs["Origin_Label"].map(to_iso3)
obs["iso_d"] = obs["Destination_Label"].map(to_iso3)

print("Transport observed rows:", len(obs))
print("ISO mapping success (origin):", obs["iso_o"].notna().mean())
print("ISO mapping success (dest):  ", obs["iso_d"].notna().mean())

# ---------- LSCI (Linear Shipping Connectivity Index) annualize & join ----------
# https://unctadstat.unctad.org/datacentre/dataviewer/US.LSCI

lsci = pd.read_csv(lsci_path)[["Economy_Label","Quarter_Label","Index_Average_Q1_2023__100_Value"]].copy()
lsci["Year"] = lsci["Quarter_Label"].str.extract(r"(20\d{2})").astype(int)
lsci = lsci.groupby(["Economy_Label","Year"], as_index=False)["Index_Average_Q1_2023__100_Value"].mean()
lsci["iso3"] = lsci["Economy_Label"].map(to_iso3)
lsci = lsci.rename(columns={"Index_Average_Q1_2023__100_Value":"lsci"})

obs2 = obs.merge(lsci.rename(columns={"iso3":"iso_o","lsci":"lsci_o"}), on=["iso_o","Year"], how="left")
obs2 = obs2.merge(lsci.rename(columns={"iso3":"iso_d","lsci":"lsci_d"}), on=["iso_d","Year"], how="left")

print("LSCI coverage origin:", obs2["lsci_o"].notna().mean())
print("LSCI coverage dest:  ", obs2["lsci_d"].notna().mean())

# ---------- LSBCI join (static) ----------
lsbci = pd.read_csv(lsbci_path)[["Economy_Label","Partner_Label","Index_Value"]].copy()
lsbci["iso_o"] = lsbci["Economy_Label"].map(to_iso3)
lsbci["iso_d"] = lsbci["Partner_Label"].map(to_iso3)
lsbci = lsbci.rename(columns={"Index_Value":"lsbci"})[["iso_o","iso_d","lsbci"]]

obs3 = obs2.merge(lsbci, on=["iso_o","iso_d"], how="left")
print("LSBCI coverage:", obs3["lsbci"].notna().mean())
print("LSBCI has time column?", any("Year" in c or "Quarter" in c for c in pd.read_csv(lsbci_path).columns))

# ---------- Distance join ----------
dist = pd.read_csv(dist_path).copy()
dist["distw"] = pd.to_numeric(dist["distw"].replace(".", np.nan), errors="coerce")
dist["dist"]  = pd.to_numeric(dist["dist"], errors="coerce")
dist["dist_km"] = dist["distw"].fillna(dist["dist"])
dist = dist[["iso_o","iso_d","dist_km"]]

obs4 = obs3.merge(dist, on=["iso_o","iso_d"], how="left")
print("Distance coverage:", obs4["dist_km"].notna().mean())

# ---------- Fuel annual availability ----------
fuel = pd.read_csv(fuel_path)
fuel_col = "VLSFO Fuel Oil, IMO 2020 Grade, 0.5%"
fuel[fuel_col] = pd.to_numeric(fuel[fuel_col].replace(r"[\$,]", "", regex=True), errors="coerce")
fuel_years = sorted(fuel["Year"].unique())
print("Fuel years:", fuel_years[:3], "...", fuel_years[-3:])
print("Fuel covers 2016-2018?", any(y in fuel_years for y in [2016,2017,2018]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 72.2 MB/s eta 0:00:00
Transport observed rows: 239499
ISO mapping success (origin): 0.9957244080351066
ISO mapping success (dest):   0.9962755585618311
LSCI coverage origin: 0.8565022237875648
LSCI coverage dest:   0.7930198461777883
LSBCI coverage: 0.6837735196295044
LSBCI has time column? False
Distance coverage: 0.8907737791022244
Fuel years: [np.int64(2019), np.int64(2020), np.int64(2021)] ... [np.int64(2024), np.int64(2025), np.int64(2026)]
Fuel covers 2016-2018? False


In [ ]:
# =========================
# Colab-ready script to transform your 4 uploaded files into:
#   transport_cost_features_2016_2021.csv.gz
#   transport_cost_features_sample.csv
#
# INPUTS (you uploaded these):
# 1) US.TransportCosts_1107_....csv      (UNCTAD transport costs, wide by year)
# 2) US.LSCI_....csv                    (UNCTAD LSCI quarterly)
# 3) US.LSBCI_....csv                   (UNCTAD LSBCI snapshot, no time col)
# 4) dist_cepii.csv                     (CEPII GeoDist dyadic)
# 5) Daily_Bunker_Fuel_Prices_....csv   (USDA bunker fuel prices)
# =========================

!pip -q install pycountry

import re
import numpy as np
import pandas as pd
import pycountry

# -------------------------
# (A) SET YOUR FILE PATHS
# -------------------------
# If you uploaded to Colab session via files.upload(), they’re usually in /content/
TRANSPORT_PATH = "/content/drive/MyDrive/is5126/US.TransportCosts_1107_20260206_072800.csv"
LSCI_PATH      = "/content/drive/MyDrive/is5126/US.LSCI_20260314_120502.csv"
LSBCI_PATH     = "/content/drive/MyDrive/is5126/US.LSBCI_20260314_120615.csv"
DIST_PATH      = "/content/drive/MyDrive/is5126/dist_cepii.csv"
FUEL_PATH      = "/content/drive/MyDrive/is5126/Daily_Bunker_Fuel_Prices_20260314.csv"

# Output files
OUT_FULL  = "/content/drive/MyDrive/is5126/transport_cost_features_2016_2021.csv.gz"
OUT_SAMPLE= "/content/drive/MyDrive/is5126/transport_cost_features_sample.csv"

# -------------------------
# (B) ISO3 mapping helper
# -------------------------
manual_iso3 = {
  'Brunei Darussalam': 'BRN',
  'Cabo Verde': 'CPV',
  'China, Hong Kong SAR': 'HKG',
  'China, Macao SAR': 'MAC',
  'China, Taiwan Province of': 'TWN',
  'Congo, Dem. Rep. of the': 'COD',
  "Cote d'Ivoire": 'CIV',
  'Curaçao': 'CUW',
  'Curacao': 'CUW',
  'Eswatini': 'SWZ',
  "Korea, Dem. People's Rep. of": 'PRK',
  "Lao People's Dem. Rep.": 'LAO',
  'Netherlands (Kingdom of the)': 'NLD',
  'Russian Federation': 'RUS',
  'State of Palestine': 'PSE',
  'Türkiye': 'TUR',
  'Turkiye': 'TUR',
  'Venezuela (Bolivarian Rep. of)': 'VEN',
  'Viet Nam': 'VNM',
  'Bolivia (Plurinational State of)': 'BOL',
  'Iran (Islamic Republic of)': 'IRN',
  'Syrian Arab Republic': 'SYR',
  'United Republic of Tanzania': 'TZA',
  'Micronesia (Federated States of)': 'FSM',
  'Dem. Rep. of the Congo': 'COD',
  'Democratic Republic of the Congo': 'COD',
  "Côte d’Ivoire": 'CIV',
  "Côte d'Ivoire": 'CIV',
}

def to_iso3(label: str):
    if pd.isna(label):
        return None
    s = str(label).strip()
    if s in manual_iso3:
        return manual_iso3[s]
    # minimal normalization
    repl = {
        "Côte d'Ivoire": "Cote d'Ivoire",
        "Côte d’Ivoire": "Cote d'Ivoire",
    }
    s2 = repl.get(s, s)
    try:
        return pycountry.countries.lookup(s2).alpha_3
    except Exception:
        return None

# -------------------------
# (C) Load + melt transport costs (wide -> long)
# -------------------------
YEAR_RE = re.compile(r"(20\d{2})_Perunit_freight_rate_USkg_Value")

t = pd.read_csv(TRANSPORT_PATH)
year_cols = [c for c in t.columns if YEAR_RE.search(c)]
if not year_cols:
    raise ValueError("No year columns matched pattern: 20xx_Perunit_freight_rate_USkg_Value")

df_long = t.melt(
    id_vars=["Origin_Label","Destination_Label","Product_Code","Product_Label"],
    value_vars=year_cols,
    var_name="YearCol",
    value_name="Freight_USD_per_kg"
)
df_long["Year"] = df_long["YearCol"].str.extract(r"(20\d{2})").astype(int)
df_long.drop(columns=["YearCol"], inplace=True)

df_obs = df_long.dropna(subset=["Freight_USD_per_kg"]).copy()
df_obs["Freight_USD_per_kg"] = pd.to_numeric(df_obs["Freight_USD_per_kg"], errors="coerce")
df_obs = df_obs.dropna(subset=["Freight_USD_per_kg"])
df_obs = df_obs[df_obs["Freight_USD_per_kg"] >= 0].copy()
df_obs["y_log"] = np.log1p(df_obs["Freight_USD_per_kg"])

# ISO codes for joining externals
df_obs["iso_o"] = df_obs["Origin_Label"].map(to_iso3)
df_obs["iso_d"] = df_obs["Destination_Label"].map(to_iso3)

print("Transport observed rows:", len(df_obs))
print("ISO origin coverage:", df_obs["iso_o"].notna().mean())
print("ISO dest coverage:  ", df_obs["iso_d"].notna().mean())

# -------------------------
# (D) LSCI: quarterly -> annual mean; join to origin + destination
# -------------------------
lsci = pd.read_csv(LSCI_PATH)

# detect value column
val_candidates = [c for c in lsci.columns if c.endswith("_Value")]
if not val_candidates:
    val_candidates = [c for c in lsci.columns if c.lower().endswith("value")]
if not val_candidates:
    raise ValueError("Could not find an LSCI value column ending with '_Value'.")

LSCI_VAL_COL = val_candidates[0]
lsci2 = lsci[["Economy_Label","Quarter_Label", LSCI_VAL_COL]].copy()
lsci2["Year"] = lsci2["Quarter_Label"].astype(str).str.extract(r"(20\d{2})").astype(int)
lsci2[LSCI_VAL_COL] = pd.to_numeric(lsci2[LSCI_VAL_COL], errors="coerce")

lsci_year = (
    lsci2.groupby(["Economy_Label","Year"], as_index=False)[LSCI_VAL_COL]
    .mean()
    .rename(columns={LSCI_VAL_COL:"lsci"})
)
lsci_year["iso3"] = lsci_year["Economy_Label"].map(to_iso3)

lsci_global_mean = float(lsci_year["lsci"].mean())

# Join to df_obs
tmp = df_obs.merge(
    lsci_year.rename(columns={"iso3":"iso_o","lsci":"lsci_o"})[["iso_o","Year","lsci_o"]],
    on=["iso_o","Year"], how="left"
).merge(
    lsci_year.rename(columns={"iso3":"iso_d","lsci":"lsci_d"})[["iso_d","Year","lsci_d"]],
    on=["iso_d","Year"], how="left"
)

print("LSCI coverage origin (pre-fill):", tmp["lsci_o"].notna().mean())
print("LSCI coverage dest   (pre-fill):", tmp["lsci_d"].notna().mean())

# fill missing
tmp["lsci_o"] = tmp["lsci_o"].fillna(lsci_global_mean)
tmp["lsci_d"] = tmp["lsci_d"].fillna(lsci_global_mean)
tmp["lsci_sum"] = tmp["lsci_o"] + tmp["lsci_d"]
tmp["lsci_gap"] = (tmp["lsci_o"] - tmp["lsci_d"]).abs()

# -------------------------
# (E) LSBCI: static snapshot (no time). Join on iso_o, iso_d
# -------------------------
lsbci = pd.read_csv(LSBCI_PATH)
LSBCI_VAL_COL = "Index_Value" if "Index_Value" in lsbci.columns else \
    ([c for c in lsbci.columns if c.lower().endswith("value")][0])

lsbci2 = lsbci[["Economy_Label","Partner_Label", LSBCI_VAL_COL]].copy()
lsbci2[LSBCI_VAL_COL] = pd.to_numeric(lsbci2[LSBCI_VAL_COL], errors="coerce")
lsbci2["iso_o"] = lsbci2["Economy_Label"].map(to_iso3)
lsbci2["iso_d"] = lsbci2["Partner_Label"].map(to_iso3)
lsbci2 = lsbci2.rename(columns={LSBCI_VAL_COL:"lsbci"})[["iso_o","iso_d","lsbci"]].dropna(subset=["iso_o","iso_d"])
lsbci2 = lsbci2.groupby(["iso_o","iso_d"], as_index=False)["lsbci"].mean()

tmp = tmp.merge(lsbci2, on=["iso_o","iso_d"], how="left")
print("LSBCI coverage (pre-fill):", tmp["lsbci"].notna().mean())
tmp["lsbci"] = tmp["lsbci"].fillna(0.0)

# -------------------------
# (F) Distance: clean, create dist_km (prefer distw), fill missing with median
# -------------------------
dist = pd.read_csv(DIST_PATH)

# convert "." to NaN then numeric
for c in ["distw","distwces","dist","distcap"]:
    if c in dist.columns:
        dist[c] = pd.to_numeric(dist[c].replace(".", np.nan), errors="coerce")

# choose best distance
if "distw" in dist.columns:
    dist["dist_km"] = dist["distw"].fillna(dist.get("dist", np.nan))
else:
    dist["dist_km"] = dist.get("dist", np.nan)

dist2 = dist[["iso_o","iso_d","dist_km"]].dropna(subset=["iso_o","iso_d"]).copy()
dist2.loc[dist2["dist_km"] <= 0, "dist_km"] = np.nan
dist_median = float(dist2["dist_km"].median())

tmp = tmp.merge(dist2, on=["iso_o","iso_d"], how="left")
print("Distance coverage (pre-fill):", tmp["dist_km"].notna().mean())

tmp["dist_km"] = tmp["dist_km"].fillna(dist_median)
tmp["log_dist"] = np.log1p(tmp["dist_km"])

# derived
tmp["lsbci_per_km"] = tmp["lsbci"] / tmp["dist_km"]

# -------------------------
# (G) Fuel: daily -> annual avg VLSFO; fill 2016-2018 with 2019 avg
# -------------------------
fuel = pd.read_csv(FUEL_PATH)

# detect VLSFO column
fuel_candidates = [c for c in fuel.columns if "VLSFO" in c]
if not fuel_candidates:
    raise ValueError("Could not find a VLSFO column in fuel CSV (expected 'VLSFO' substring).")
FUEL_COL = fuel_candidates[0]

fuel[FUEL_COL] = pd.to_numeric(fuel[FUEL_COL].replace(r"[\$,]", "", regex=True), errors="coerce")
fuel_year = (
    fuel.groupby("Year", as_index=False)[FUEL_COL].mean()
    .rename(columns={FUEL_COL:"fuel_vlsfo_avg"})
)

# Fill 2016-2018 using 2019 average (Option B)
fuel_2019 = fuel_year.loc[fuel_year["Year"] == 2019, "fuel_vlsfo_avg"]
fuel_fill = float(fuel_2019.iloc[0]) if len(fuel_2019) else float(fuel_year["fuel_vlsfo_avg"].mean())

for y in [2016, 2017, 2018]:
    if y not in set(fuel_year["Year"]):
        fuel_year = pd.concat([fuel_year, pd.DataFrame({"Year":[y], "fuel_vlsfo_avg":[fuel_fill]})], ignore_index=True)

fuel_year = fuel_year.sort_values("Year").reset_index(drop=True)

tmp = tmp.merge(fuel_year, on="Year", how="left")
tmp["fuel_vlsfo_avg"] = tmp["fuel_vlsfo_avg"].fillna(fuel_fill)

print("Fuel years present after fill:", sorted(tmp["Year"].unique().tolist()))

# -------------------------
# (H) Final sanity checks + export
# -------------------------
# No key duplication should occur
dup_rate = tmp.duplicated(subset=["Origin_Label","Destination_Label","Product_Code","Year"]).mean()
print("Duplicate rate on (Origin,Dest,Product,Year):", dup_rate)

# Quick coverage after fills (should be 0 missing for these)
check_cols = ["lsci_o","lsci_d","lsbci","dist_km","fuel_vlsfo_avg"]
print("Any NaN left in core engineered features?", tmp[check_cols].isna().any().any())

# Export full + sample (no parquet needed)
tmp.to_csv(OUT_FULL, index=False, compression="gzip")
tmp.sample(2000, random_state=42).to_csv(OUT_SAMPLE, index=False)

print("\n✅ Exported:")
print("Full:", OUT_FULL)
print("Sample:", OUT_SAMPLE)
print("\nTip: In Colab, download with:")
print("from google.colab import files; files.download(OUT_FULL)")

Transport observed rows: 239499
ISO origin coverage: 0.985148163457885
ISO dest coverage:   0.987966546833181
LSCI coverage origin (pre-fill): 0.8394877699295828
LSCI coverage dest   (pre-fill): 0.7854495187015228
LSBCI coverage (pre-fill): 0.5828890902069205
Distance coverage (pre-fill): 0.7941262456553203
Fuel years present after fill: [2016, 2017, 2018, 2019, 2020, 2021]
Duplicate rate on (Origin,Dest,Product,Year): 0.1355792725842137
Any NaN left in core engineered features? False

✅ Exported:
Full: /content/drive/MyDrive/is5126/transport_cost_features_2016_2021.csv.gz
Sample: /content/drive/MyDrive/is5126/transport_cost_features_sample.csv

Tip: In Colab, download with:
from google.colab import files; files.download(OUT_FULL)


**Full Table after merging features**


```
Origin_Label
Destination_Label
Product_Code
Product_Label
Freight_USD_per_kg
Year
y_log
iso_o
iso_d
lsci_o
lsci_d
lsci_sum
lsci_gap
lsbci
dist_km
log_dist
lsbci_per_km
fuel_vlsfo_avg
miss_lsci_o
miss_lsci_d
miss_lsbci
miss_dist
miss_fuel
```

**Date range after merge**
Year range: 2016 → 2021 (inclusive)


### 3 Evaluation Splits
### Cold-route (primary match)
What it simulates:
A new trade deal: the country pair (Origin, Destination) was never seen in training (or you treat it as unseen), and you want to estimate what it would cost if they started shipping.

How we test it:
We hold out entire origin–destination routes (all products/years for those routes) during training, then predict on those unseen routes.

Why it matters:
This tests whether your model can generalize using structural features like:

distance

LSCI / LSBCI

fuel/year effects
rather than memorizing known lanes.


### Impute (secondary match, supports completeness)
Your route planner also needs a dense cost matrix so it can compute best paths:

many country pairs will have missing costs for some years

imputation fills these gaps so the planner can run without “holes”

So impute matches the “fill missing edges / missing years” part of building a usable routing graph.

But impute assumes the lane has at least some history, so it’s less “new deal” than cold-route.

What it simulates:
You already have a trade relationship for a specific (Origin, Destination, Product), but some years are missing (no reported transport cost). You want to fill in the missing cells.

How we test it:
For each (O,D,P), we hide some observed years but keep at least one year in training, then predict the hidden ones.

Why it matters:
This is the most “pure” test of your stated goal: estimate missing transport costs when there is partial history.

### Time split (optional / stretch goal)

Time split matters only if your route planner is positioned as:

“Forecast next year’s shipping cost under changing market conditions.”

If tariffs change today, you’re mostly doing scenario analysis (tariff rate change) + current year cost estimation, not necessarily forecasting future freight regimes.

So time split is nice for maturity, but it’s not the core requirement for a tariff scenario planner.

What it simulates:
You train on past years (e.g., 2016–2020) and you want to estimate costs in a future year (e.g., 2021).

How we test it:
Train on 2016–2020, test on 2021 (and we used a time-aware validation: 2016–2019 train, 2020 val, 2021 test).

Why it matters:
It checks if your model is stable under year-to-year changes (market shifts, shocks), which is the hardest setting.

In [ ]:
# =========================
# Colab notebook cell (FULL):
# 1) Load engineered dataset from Google Drive
# 2) Build 3 evaluation splits (Impute / Cold-route / Time)
# 3) Train leak-free CatBoost (with early stopping)
# 4) Use ALL engineered external features + history + backoff aggregates
# 5) Safe log->cost conversion (prevents overflow) + robust metrics
# =========================

!pip -q install catboost pycountry

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

# -------------------------
# 0) Mount Drive + set path
# -------------------------
from google.colab import drive
drive.mount("/content/drive")

# CHANGE THIS to where you saved the .csv.gz in Drive
DATA_PATH = "/content/drive/MyDrive/is5126/transport_cost_features_2016_2021.csv.gz"


# -------------------------
# 1) Load engineered dataset
# -------------------------
df = pd.read_csv(DATA_PATH, compression="gzip")

# Safety filters
df = df.dropna(subset=["Freight_USD_per_kg", "Origin_Label", "Destination_Label", "Product_Code", "Year"]).copy()
df["Freight_USD_per_kg"] = pd.to_numeric(df["Freight_USD_per_kg"], errors="coerce")
df = df.dropna(subset=["Freight_USD_per_kg"]).copy()
df = df[df["Freight_USD_per_kg"] >= 0].copy()

df["Year"] = df["Year"].astype(int)
df["Product_Code"] = df["Product_Code"].astype(str)  # keep categorical stable
df["y_log"] = np.log1p(df["Freight_USD_per_kg"])

df["Route"] = df["Origin_Label"].astype(str) + "|" + df["Destination_Label"].astype(str)
df["ODP"]   = df["Route"] + "|" + df["Product_Code"].astype(str)

print("Loaded rows:", len(df), "cols:", df.shape[1])
print("Years:", sorted(df["Year"].unique()))
print("Products:", sorted(df["Product_Code"].unique())[:10], "...")


# -------------------------
# 2) Evaluation splits
# -------------------------
def split_mask_within_odp(df_obs: pd.DataFrame, mask_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    key = ["Origin_Label","Destination_Label","Product_Code"]
    test_idx = []
    for _, g in df_obs.groupby(key):
        idx = g.index.values
        n = len(idx)
        if n <= 1:
            continue
        k = max(1, int(round(mask_frac * n)))
        k = min(k, n-1)  # keep at least 1 in train
        test_idx.append(rng.choice(idx, size=k, replace=False))
    test_idx = np.concatenate(test_idx) if test_idx else np.array([], dtype=int)
    test = df_obs.loc[test_idx]
    train = df_obs.drop(index=test_idx)
    return train, test

def split_cold_routes(df_obs: pd.DataFrame, test_size=0.2, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx_train, idx_test = next(gss.split(df_obs, groups=df_obs["Route"]))
    return df_obs.iloc[idx_train], df_obs.iloc[idx_test]

def split_time(df_obs: pd.DataFrame, train_years=(2016, 2020), test_year=2021):
    train = df_obs[(df_obs["Year"]>=train_years[0]) & (df_obs["Year"]<=train_years[1])]
    test  = df_obs[df_obs["Year"]==test_year]
    return train, test


# -------------------------
# 3) Train-only backoff + history features (no leakage)
# -------------------------
def build_train_aggregates(train_base: pd.DataFrame):
    aggs = {}
    aggs["global_mean"] = float(train_base["Freight_USD_per_kg"].mean())

    aggs["origin_year"] = (
        train_base.groupby(["Origin_Label","Year"])["Freight_USD_per_kg"]
        .mean().rename("origin_year_mean").reset_index()
    )
    aggs["dest_year"] = (
        train_base.groupby(["Destination_Label","Year"])["Freight_USD_per_kg"]
        .mean().rename("dest_year_mean").reset_index()
    )
    aggs["product_year"] = (
        train_base.groupby(["Product_Code","Year"])["Freight_USD_per_kg"]
        .mean().rename("product_year_mean").reset_index()
    )
    aggs["route"] = (
        train_base.groupby(["Origin_Label","Destination_Label"])["Freight_USD_per_kg"]
        .mean().rename("route_mean").reset_index()
    )
    aggs["year"] = (
        train_base.groupby(["Year"])["Freight_USD_per_kg"]
        .mean().rename("global_year_mean").reset_index()
    )
    return aggs

def attach_train_aggregates(df_in: pd.DataFrame, aggs: dict):
    out = df_in.copy()
    out = out.merge(aggs["origin_year"], on=["Origin_Label","Year"], how="left")
    out = out.merge(aggs["dest_year"], on=["Destination_Label","Year"], how="left")
    out = out.merge(aggs["product_year"], on=["Product_Code","Year"], how="left")
    out = out.merge(aggs["route"], on=["Origin_Label","Destination_Label"], how="left")
    out = out.merge(aggs["year"], on=["Year"], how="left")

    g = aggs["global_mean"]
    for c in ["origin_year_mean","dest_year_mean","product_year_mean","route_mean","global_year_mean"]:
        out[c] = out[c].fillna(g)
    return out

def build_train_history(train_base: pd.DataFrame):
    key = ["Origin_Label","Destination_Label","Product_Code"]

    stats = (
        train_base.groupby(key)["Freight_USD_per_kg"]
        .agg(hist_mean="mean", hist_std="std", hist_count="count")
        .reset_index()
    )
    stats["hist_std"] = stats["hist_std"].fillna(0.0)

    yearly = (
        train_base.groupby(key + ["Year"], as_index=False)["Freight_USD_per_kg"]
        .mean()
        .sort_values(key + ["Year"])
    )
    yearly["lag1"] = yearly.groupby(key)["Freight_USD_per_kg"].shift(1)
    lag_map = yearly[key + ["Year","lag1"]]
    return stats, lag_map

def attach_train_history(df_in: pd.DataFrame, stats: pd.DataFrame, lag_map: pd.DataFrame, aggs: dict):
    key = ["Origin_Label","Destination_Label","Product_Code"]
    out = df_in.copy()
    out = out.merge(stats, on=key, how="left")
    out = out.merge(lag_map, on=key + ["Year"], how="left")

    out["hist_std"] = out["hist_std"].fillna(0.0)
    out["hist_count"] = out["hist_count"].fillna(0)

    g = aggs["global_mean"]
    # fallback for unseen ODP: use product-year mean if available else global mean
    if "product_year_mean" in out.columns:
        out["hist_mean"] = out["hist_mean"].fillna(out["product_year_mean"]).fillna(g)
    else:
        out["hist_mean"] = out["hist_mean"].fillna(g)

    return out


# -------------------------
# 4) Safe prediction + metrics
# -------------------------
def safe_expm1_from_log(pred_log, ylog_train, pad=1.0):
    lo, hi = np.quantile(ylog_train, [0.001, 0.999])
    lo -= pad; hi += pad
    pred_log_clip = np.clip(pred_log, lo, hi)
    pred = np.expm1(pred_log_clip)
    pred = np.clip(pred, 0.0, None)
    return pred, {
        "PredClip_lo_ylog": float(lo),
        "PredClip_hi_ylog": float(hi),
        "PctPredClipped": float(np.mean(pred_log != pred_log_clip))
    }

def safe_metrics(true, pred):
    true = np.asarray(true, float)
    pred = np.asarray(pred, float)
    true = np.clip(true, 0.0, None)
    pred = np.clip(pred, 0.0, None)

    err = np.abs(true - pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    cut = np.quantile(err, 0.99)
    m = err <= cut
    rmse_trim = np.sqrt(np.mean((true[m] - pred[m])**2)) if np.any(m) else rmse

    return {
        "MAE_USDkg": float(mean_absolute_error(true, pred)),
        "MedAE_USDkg": float(median_absolute_error(true, pred)),
        "RMSE_USDkg": float(rmse),
        "LogMAE": float(np.mean(np.abs(np.log1p(true) - np.log1p(pred)))),
        "P95_AE_USDkg": float(np.quantile(err, 0.95)),
        "TrimmedRMSE_99": float(rmse_trim),
    }


# -------------------------
# 5) Training function using ALL engineered external features
# -------------------------
EXTERNAL_FEATURES = [
    "log_dist", "dist_km",
    "lsci_o", "lsci_d", "lsci_sum", "lsci_gap",
    "lsbci", "lsbci_per_km",
    "fuel_vlsfo_avg"
]

def train_catboost_with_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    use_history_features: bool = True,
    seed: int = 42,
    val_frac: float = 0.1,
    iterations: int = 5000,
    learning_rate: float = 0.05,
    depth: int = 8,
    loss_function: str = "Huber:delta=1.0",
    early_stopping_rounds: int = 200,
    verbose: int = 200,
    clip_pad: float = 1.0,
):
    tr_inner, val_df = train_test_split(train_df, test_size=val_frac, random_state=seed)
    base = tr_inner.copy()

    # train-only aggregates
    aggs = build_train_aggregates(base)
    tr_f  = attach_train_aggregates(tr_inner, aggs)
    val_f = attach_train_aggregates(val_df, aggs)
    te_f  = attach_train_aggregates(test_df, aggs)

    # train-only history
    if use_history_features:
        stats, lag_map = build_train_history(base)
        tr_f  = attach_train_history(tr_f,  stats, lag_map, aggs)
        val_f = attach_train_history(val_f, stats, lag_map, aggs)
        te_f  = attach_train_history(te_f,  stats, lag_map, aggs)

        feats = [
            "Origin_Label","Destination_Label","Product_Code",
            "Year",
            "origin_year_mean","dest_year_mean","product_year_mean","route_mean","global_year_mean",
            "hist_mean","hist_std","hist_count","lag1",
        ]
    else:
        feats = [
            "Origin_Label","Destination_Label","Product_Code",
            "Year",
            "origin_year_mean","dest_year_mean","product_year_mean","route_mean","global_year_mean",
        ]

    # Add engineered externals IF present
    for c in EXTERNAL_FEATURES:
        if c in tr_f.columns and c in val_f.columns and c in te_f.columns:
            feats.append(c)

    # categoricals (Year numeric)
    cat_feats = ["Origin_Label","Destination_Label","Product_Code"]

    tr_pool  = Pool(tr_f[feats],  tr_f["y_log"],  cat_features=cat_feats)
    val_pool = Pool(val_f[feats], val_f["y_log"], cat_features=cat_feats)
    te_pool  = Pool(te_f[feats],  te_f["y_log"],  cat_features=cat_feats)

    model = CatBoostRegressor(
        loss_function=loss_function,
        depth=depth,
        learning_rate=learning_rate,
        iterations=iterations,
        random_seed=seed,
        verbose=verbose,
    )

    model.fit(
        tr_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds,
    )

    pred_log = model.predict(te_pool)
    pred, clip_info = safe_expm1_from_log(pred_log, tr_f["y_log"].values, pad=clip_pad)

    true = te_f["Freight_USD_per_kg"].values
    out = safe_metrics(true, pred)
    out.update(clip_info)
    out["n_test"] = int(len(te_f))
    out["features_used"] = len(feats)
    return model, out


# -------------------------
# 6) Run the 3 evaluations with external features included
# -------------------------
# (A) Imputation realism
tr, te = split_mask_within_odp(df, mask_frac=0.2, seed=42)
_, metrics_impute = train_catboost_with_features(tr, te, use_history_features=True, verbose=200)

# (B) Cold-route realism
tr, te = split_cold_routes(df, test_size=0.2, seed=42)
_, metrics_cold = train_catboost_with_features(tr, te, use_history_features=False, verbose=200)

# (C) Time split realism
tr, te = split_time(df, train_years=(2016, 2020), test_year=2021)
_, metrics_time = train_catboost_with_features(tr, te, use_history_features=True, verbose=200)

print("IMPUTE (mask-within-ODP):", metrics_impute)
print("COLD ROUTE:", metrics_cold)
print("TIME SPLIT:", metrics_time)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded rows: 277063 cols: 20
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Products: ['2106', '3304', '6109', '8517', '9404'] ...
0:	learn: 0.5369438	test: 0.5702148	best: 0.5702148 (0)	total: 281ms	remaining: 23m 26s
200:	learn: 0.0589721	test: 0.1321306	best: 0.1320779 (198)	total: 1m 1s	remaining: 24m 26s
400:	learn: 0.0469970	test: 0.1274056	best: 0.1274056 (400)	total: 1m 57s	remaining: 22m 23s
600:	learn: 0.0423536	test: 0.1252024	best: 0.1252024 (600)	total: 2m 48s	remaining: 20m 32s
800:	learn: 0.0394473	test: 0.1236233	best: 0.1236192 (798)	total: 3m 46s	remaining: 19m 45s
1000:	learn: 0.0371660	test: 0.1221478	best: 0.1221478 (1000)	total: 4m 37s	remaining: 18m 28s
1200:	learn: 0.0353236	test: 0.1211826	best: 0.1211676 (1197)	total: 5m 29s	remaining: 17m 22s
1400:	learn: 0.0336747	test: 0.120

In [ ]:
df["year_idx"] = df["Year"] - 2016
EXTERNAL_FEATURES.append("year_idx")

In [ ]:
def train_time_split_model(df_train_2016_2020, df_test_2021, use_history_features=True, verbose=200):
    # time-aware validation
    tr_inner = df_train_2016_2020[df_train_2016_2020["Year"] <= 2019].copy()
    val_df   = df_train_2016_2020[df_train_2016_2020["Year"] == 2020].copy()

    base = tr_inner.copy()
    aggs = build_train_aggregates(base)
    tr_f  = attach_train_aggregates(tr_inner, aggs)
    val_f = attach_train_aggregates(val_df, aggs)
    te_f  = attach_train_aggregates(df_test_2021, aggs)

    if use_history_features:
        stats, lag_map = build_train_history(base)
        tr_f  = attach_train_history(tr_f,  stats, lag_map, aggs)
        val_f = attach_train_history(val_f, stats, lag_map, aggs)
        te_f  = attach_train_history(te_f,  stats, lag_map, aggs)

        feats = [
            "Origin_Label","Destination_Label","Product_Code",
            "Year",
            "origin_year_mean","dest_year_mean","product_year_mean","route_mean","global_year_mean",
            "hist_mean","hist_std","hist_count","lag1",
        ]
    else:
        feats = [
            "Origin_Label","Destination_Label","Product_Code",
            "Year",
            "origin_year_mean","dest_year_mean","product_year_mean","route_mean","global_year_mean",
        ]

    for c in EXTERNAL_FEATURES:
        if c in tr_f.columns:
            feats.append(c)

    cat_feats = ["Origin_Label","Destination_Label","Product_Code"]

    tr_pool  = Pool(tr_f[feats],  tr_f["y_log"],  cat_features=cat_feats)
    val_pool = Pool(val_f[feats], val_f["y_log"], cat_features=cat_feats)
    te_pool  = Pool(te_f[feats],  te_f["y_log"],  cat_features=cat_feats)

    model = CatBoostRegressor(
        loss_function="Huber:delta=1.0",
        depth=8,
        learning_rate=0.05,
        iterations=5000,
        random_seed=42,
        verbose=verbose,
    )
    model.fit(tr_pool, eval_set=val_pool, use_best_model=True, early_stopping_rounds=200)

    pred_log = model.predict(te_pool)
    pred, clip_info = safe_expm1_from_log(pred_log, tr_f["y_log"].values, pad=1.0)
    true = te_f["Freight_USD_per_kg"].values

    out = safe_metrics(true, pred)
    out.update(clip_info)
    out["n_test"] = int(len(te_f))
    out["features_used"] = len(feats)
    return model, out

In [ ]:
tr_2016_2020, te_2021 = split_time(df, train_years=(2016,2020), test_year=2021)
_, metrics_time_fixed = train_time_split_model(tr_2016_2020, te_2021, use_history_features=True, verbose=200)
print("TIME SPLIT (time-aware val):", metrics_time_fixed)

0:	learn: 0.6523539	test: 0.5937896	best: 0.5937896 (0)	total: 255ms	remaining: 21m 15s
200:	learn: 0.0368032	test: 0.1427816	best: 0.1417280 (161)	total: 50.3s	remaining: 20m 1s
400:	learn: 0.0310556	test: 0.1383046	best: 0.1382237 (392)	total: 1m 43s	remaining: 19m 47s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.1382236553
bestIteration = 392

Shrink model to first 393 iterations.
TIME SPLIT (time-aware val): {'MAE_USDkg': 8.885875654239278, 'MedAE_USDkg': 1.3686429537233904, 'RMSE_USDkg': 167.35389292423935, 'LogMAE': 0.6133147041826412, 'P95_AE_USDkg': 34.626336771643004, 'TrimmedRMSE_99': 13.77805439895468, 'PredClip_lo_ylog': -0.982160081871669, 'PredClip_hi_ylog': 6.982423256283772, 'PctPredClipped': 0.0, 'n_test': 46335, 'features_used': 23}



### Cold-route (new deal) — performing well

MAE: 5.31

MedAE: 0.82

LogMAE: 0.42

P95 AE: 18.61

TrimmedRMSE_99: 9.25

This is your strongest business result because it improved a lot after adding distance/LSCI/LSBCI/fuel, and the typical error (MedAE) is low.

### Impute (fill missing years) — performing well for “typical” cases

MAE: 5.63

MedAE: 0.42 ✅ very good

LogMAE: 0.32 ✅ best among the three

P95 AE: 19.78

TrimmedRMSE_99: 9.19 ✅ good

This means: for most missing cells, your imputation is accurate. The remaining weakness is a small number of extreme spikes (RMSE is still large), but your robust metrics show it’s strong for the bulk.


⚠️ Mixed / hardest case

### Time split (forecast) — acceptable but not “strong”

With time-aware validation:

MAE: 8.89

MedAE: 1.37 ✅ ok

LogMAE: 0.61 ✅ reasonable

P95 AE: 34.63 ⚠️ higher

TrimmedRMSE_99: 13.78 ⚠️ higher

RMSE still huge because of rare 2021 spikes

## Next steps:

### Lock the “final” modeling setup (so results are stable)

Keep Cold-route as your headline metric (route planner “new deal” scenario).

Keep Impute as supporting metric (dense cost matrix).

Keep Time split as optional (forecasting), with the time-aware validation version.

Deliverable: a small table in your notebook/report with the 3 metrics and a sentence each.

### Add P50 + P90 models (this is the best upgrade for a planner)

Right now you have one point estimate (Huber). For route planning under uncertainty, train:

P50: Quantile:alpha=0.5 (typical cost)

P90: Quantile:alpha=0.9 (risk buffer)

Then your route planner can show:

“Cheapest route by expected cost (P50)”

“Safer route by worst-plausible cost (P90)”

“Risk premium = P90 − P50”

This also gives you a strong “tariff + risk” narrative.

### Generate a complete cost table for a chosen year (for routing)

Pick a year to demo (e.g., 2021 or latest year available).
Create a table for every (origin, destination, product):

If observed → use observed USD/kg

If missing → use model prediction (P50; optionally also P90)

Deliverable: cost_matrix_{product}_{year}.csv with columns:
origin_iso3, dest_iso3, product_code, cost_p50, cost_p90, source(observed/predicted)

### Build the route planner graph + K-shortest paths

For each product+year:

Nodes = countries (ISO3)

Edge weight = predicted freight cost (P50 or P90)

Run:

shortest path (Dijkstra)

optionally K-shortest paths (top 3 alternatives)

constraints: max hops (e.g., ≤2 transshipments)

Deliverable: function recommend_routes(origin, dest, product, year, risk='p50'/'p90', max_hops=2).

### Add tariffs as a scenario layer (simple but credible)

Tariff cost needs:

HS code (you already have)

origin & destination (you have)

tariff rate (scenario input)

For the MVP:

Create a small tariff scenario table (even manual) like:

baseline: 0%

scenario A: +10% for HS 8517 from CHN→USA

scenario B: +25% for HS 6109 from … etc.


Then compute: Total Landed Cost
=
FreightCost
+
(
TariffRate
×
DeclaredValue
)
Total Landed Cost=FreightCost+(TariffRate×DeclaredValue)

Deliverable: route ranking by total landed cost, not just freight.

### MVP UI (Streamlit is easiest)

Inputs:

origin country

destination country

HS code (one of the 5)

year

declared value

tariff scenario dropdown

risk toggle: P50 vs P90
Outputs:

Top 3 routes with:

freight cost P50/P90

tariff cost

total landed cost

number of hops